# This script is used to test an OpenAI model

In [1]:
import os
from openai import OpenAI
from datetime import datetime
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
import json
import sys
sys.path.append('..')
import helper
from utils import clean_response_bool, parse_response
import numpy as np
from datetime import datetime
import glob

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Load OPENAI_API_KEY from .env file
load_dotenv()

client = OpenAI()

In [2]:
TEST_PROMPTS = "../../prompts/domain_promts.json"
MODEL_NAME = "gpt-4.1-mini-2025-04-14"
OPENAI_TEST_FILE_ID = "file-Q5mRbt7ejvoDEPLkJ8hVKN"
TEST_SET_PATH = "/ceph/aasteine/fine-tuning-paper/data/wdc/wdcproducts80cc20rnd050un_test_gs.pkl"

In [3]:
def insert_product_descriptions(prompt_template: str, product1: str, product2: str):
    # Replace placeholder texts with actual product descriptions
    prompt = prompt_template.replace("'Entity 1'", product1).replace("'Entity 2'", product2)
    return prompt

In [4]:
def serialize_product(row, side='left'):
    """Serialize product attributes from a row, only including non-NaN values.
    
    Args:
        row: DataFrame row containing product information
        side: 'left' or 'right' to indicate which product to serialize
        
    Returns:
        str: Serialized product string with non-NaN attributes
    """
    attributes = []
    
    # Add brand if available
    brand = row[f'brand_{side}']
    if pd.notna(brand):
        attributes.append(f"[BRAND] {brand}")
    
    # Add title (required)
    title = row[f'title_{side}']
    if pd.notna(title):
        attributes.append(f"[TITLE] {title}")
    
    # Add description if available
    description = row[f'description_{side}']
    if pd.notna(description):
        attributes.append(f"[DESCRIPTION] {description}")
    
    # Add price and currency if available
    price = row[f'price_{side}']
    if pd.notna(price):
        price_str = f"[PRICE] {price}"
        currency = row[f'priceCurrency_{side}']
        if pd.notna(currency):
            price_str += f" [CURRENCY] {currency}"
        attributes.append(price_str)
    
    return " ".join(attributes)


In [5]:
def create_prompt(prompt, custom_id, model, product_1=None, product_2=None):
    if product_1 is not None and product_2 is not None:
        prompt = insert_product_descriptions(prompt, product_1, product_2)
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model,
            "messages": [
                {"role": "user", "content": prompt},
            ],
            "max_tokens": 5,
            "temperature": 0
        }
    }

In [6]:
def create_batch_job(test_dir:str, test_set_path:str, test_prompts_path:str):
    # Load the test set
    test_set = pd.read_pickle(test_set_path)

    # open fine-tune run config
    with open(os.path.join(test_dir, "fine-tune-run_config.json"), "r") as f:
        run_config = json.load(f)
        
    # get the name of the fine-tuned model via the openai api
    fine_tune_model = client.fine_tuning.jobs.retrieve(run_config['job_id']).fine_tuned_model

    # Create output directory structure
    run_name = run_config['run_name']
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Load all prompts we want to test
    with open(test_prompts_path, 'r') as file:
        prompts = json.load(file)

    batch_job = []
    print(f"Creating batch job for {run_name}. The batch will contain {len(prompts)} prompts. With {len(test_set)} pairs each.")
    for task in prompts:
        prompt_template = task['prompt']
        prompt_id = task['id']

        for index, row in test_set.iterrows():
            product1, product2 = row['title_left'], row['title_right']
            label = row.get('label') 
            pair_id = row['pair_id']
            
            custom_id = f"{prompt_id};{pair_id};{label}"
            prompt = create_prompt(prompt_template, custom_id, fine_tune_model, product1, product2)
            batch_job.append(prompt)
            

    # Save the input file
    input_file_path = os.path.join(test_dir, f"testing_{run_name}_input.jsonl")
    with open(input_file_path, "w") as f:
        for request in batch_job:
            f.write(json.dumps(request) + "\n")
            
    # upload the batch file to openai
    batch_input_file = client.files.create(
        file=open(input_file_path, "rb"),
        purpose="batch"
    )

    # create the batch
    batch = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"description": run_name}
    )

    # Save the run configuration
    run_config = {
        "run_name": run_name,
        "timestamp": timestamp,
        "model": fine_tune_model,
        "batch_id": batch.id,
        "input_file_id": batch_input_file.id,
        "endpoint": "/v1/chat/completions",
        "completion_window": "24h",
        "metadata": batch.metadata,
        "created_at": batch.created_at,
        "status": batch.status
    }

    with open(os.path.join(test_dir, "test_run_config.json"), "w") as f:
        json.dump(run_config, f, indent=2)

    print(f"Run configuration and input files saved to: {test_dir}")
    print(f"Batch ID: {batch.id}")


In [7]:
def get_batch_results(testing_dir):
    with open(os.path.join(testing_dir, "test_run_config.json"), "r") as f:
        run_config = json.load(f)
    
    batch_id = run_config['batch_id']

    # Retrieve the batch
    batch = client.batches.retrieve(batch_id)
    print(f"Batch status: {batch.status}")
    print(f"Created at: {datetime.utcfromtimestamp(batch.created_at).strftime('%Y-%m-%d %H:%M:%S')}")

    # Check if the batch has completed and has an output file
    if batch.output_file_id:
        # Download the output file content
        file_content = client.files.content(batch.output_file_id)
        
        # Write the content to a .jsonl file
        output_file = os.path.join(testing_dir, f"batch_output_{batch_id}.jsonl")
        with open(output_file, "w") as f:
            f.write(file_content.text)
        
        print(f"Batch results saved to: {output_file}")
    else:
        print("Batch is not completed or output file is not available.")
    return output_file

In [8]:
def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    
    return accuracy, f1, precision, recall

In [9]:
def get_batch_results_and_calculate_metrics(testing_dir):
    # Load the result file 
    result_path = get_batch_results(testing_dir)
    gpt_result = pd.read_json(result_path, lines=True)


    # Split the custom_id into dataset, task, pair_id, and label
    gpt_result[['task', 'pair_id', 'label']] = gpt_result.custom_id.str.split(";", expand=True)
    gpt_result = gpt_result.drop(columns=['custom_id'])

    # Apply the parse_response function to the response column
    parsed_df = gpt_result["response"].apply(parse_response)

    # Concatenate the parsed results with the original DataFrame
    gpt_result = pd.concat([gpt_result, parsed_df], axis=1)

    # Transform 'content' to binary (0 or 1 based on "Yes")
    gpt_result['content'] = gpt_result['content'].apply(clean_response_bool)

    # Convert label from string to integer
    gpt_result['label'] = gpt_result['label'].astype(int)

    # Group by 'dataset' and 'task', then calculate metrics
    results = []
    grouped = gpt_result.groupby(['task'])

    for (task), group in grouped:
        y_true = group['label']
        y_pred = group['content']
        
        accuracy, f1, precision, recall = calculate_metrics(y_true, y_pred)
        
        results.append({
            'task': task,
            'accuracy': accuracy,
            'f1_score': f1,
            'precision': precision,
            'recall': recall
        })

    # Convert the results into a DataFrame
    metrics_df = pd.DataFrame(results)

    # Save metrics to CSV in the same directory
    output_path = os.path.join(testing_dir, 'testing_stats.csv')
    metrics_df.to_csv(output_path, index=False)
    print(f"Metrics saved to {output_path}")
    print(f"Best performing task: {metrics_df.sort_values(by='f1_score', ascending=False).iloc[0]['f1_score']}")
    return output_path

## Zero shot

In [11]:
# Load the test set
test_set = pd.read_pickle("/ceph/aasteine/fine-tuning-paper/data/wdc/wdcproducts80cc20rnd050un_test_gs.pkl")

# Create output directory structure
run_name = "zero-shot"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../../results/{MODEL_NAME}/all-attributes/{run_name}/{timestamp}"
os.makedirs(output_dir, exist_ok=True)

# Load all prompts we want to test
with open(TEST_PROMPTS, 'r') as file:
    prompts = json.load(file)

batch_job = []
print(f"Creating batch job for {run_name}. The batch will contain {len(prompts)} prompts. With {len(test_set)} pairs each.")
for task in prompts:
    title = task['title']
    prompt_template = task['prompt']
    prompt_id = task['id']

    for index, row in test_set.iterrows():
        product_1 = serialize_product(row, "left")
        product_2 = serialize_product(row, "right")
        label = row.get('label') 
        pair_id = row['pair_id']
        
        custom_id = f"{prompt_id};{pair_id};{label}"
        prompt = create_prompt(prompt_template, custom_id, MODEL_NAME, product_1, product_2)
        batch_job.append(prompt)
        

# Save the input file
input_file_path = os.path.join(output_dir, "input.jsonl")
with open(input_file_path, "w") as f:
    for request in batch_job:
        f.write(json.dumps(request) + "\n")
        
# upload the batch file to openai
batch_input_file = client.files.create(
    file=open(input_file_path, "rb"),
    purpose="batch"
)

# create the batch
batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={"description": run_name}
)

# Save the run configuration
run_config = {
    "run_name": run_name,
    "timestamp": timestamp,
    "model": MODEL_NAME,
    "batch_id": batch.id,
    "input_file_id": batch_input_file.id,
    "endpoint": "/v1/chat/completions",
    "completion_window": "24h",
    "metadata": batch.metadata,
    "created_at": batch.created_at,
    "status": batch.status
}

with open(os.path.join(output_dir, "test_run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print(f"Run configuration and input files saved to: {output_dir}")
print(f"Batch ID: {batch.id}")


Creating batch job for zero-shot. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/zero-shot/20250509_151138
Batch ID: batch_681dff12552881908021df56a3e2c0ac


In [10]:
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/all-attributes/zero-shot/20250509_151138")

Batch status: completed
Created at: 2025-05-09 13:11:46
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/zero-shot/20250509_151138/batch_output_batch_681dff12552881908021df56a3e2c0ac.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/all-attributes/zero-shot/20250509_151138/testing_stats.csv
Best performing task: 0.8438978240302744


'../../results/gpt-4.1-mini-2025-04-14/all-attributes/zero-shot/20250509_151138/testing_stats.csv'

## WDC no augmentation

In [11]:
create_batch_job("../../results/gpt-4.1-mini-2025-04-14/all-attributes/no_augmentation/20250509_150226", TEST_SET_PATH, TEST_PROMPTS)

Creating batch job for wdc_all_attributes_no_augmentation. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/no_augmentation/20250509_150226
Batch ID: batch_681f193a51e88190b5ea26046b0f2af3


In [14]:
# Load the result file 
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809")

Batch status: completed
Created at: 2025-05-08 18:28:47
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809/batch_output_batch_681cf7dfec08819091db0cd4b4499fe4.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809/testing_stats.csv
Best performing task: 0.8207885304659498


'../../results/gpt-4.1-mini-2025-04-14/fine-tune-wdc-small-regular/20250507_173809/testing_stats.csv'

## WDC explanations

In [12]:
create_batch_job("../../results/gpt-4.1-mini-2025-04-14/all-attributes/explanation/20250509_150722", TEST_SET_PATH, TEST_PROMPTS)

Creating batch job for wdc_all_attributes_explanation. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/explanation/20250509_150722
Batch ID: batch_681f1957061c819092914cc30f121fdd


In [14]:
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/all-attributes/explanation/20250509_150722")

Batch status: completed
Created at: 2025-05-10 09:16:07
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/explanation/20250509_150722/batch_output_batch_681f1957061c819092914cc30f121fdd.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/all-attributes/explanation/20250509_150722/testing_stats.csv
Best performing task: 0.8053333333333333


'../../results/gpt-4.1-mini-2025-04-14/all-attributes/explanation/20250509_150722/testing_stats.csv'

## Simple Swapping

In [16]:
create_batch_job("../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-all-attributes/20250510_112703", TEST_SET_PATH, TEST_PROMPTS)

Creating batch job for fine-tune-wdc-simple-swapping-all-attributes. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-all-attributes/20250510_112703
Batch ID: batch_681f2dd8c2b881909709754b3764270c


In [10]:
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-all-attributes/20250510_112703")

Batch status: completed
Created at: 2025-05-10 10:43:36
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-all-attributes/20250510_112703/batch_output_batch_681f2dd8c2b881909709754b3764270c.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-all-attributes/20250510_112703/testing_stats.csv
Best performing task: 0.8299065420560747


'../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-all-attributes/20250510_112703/testing_stats.csv'

## 10% swapping


In [17]:
create_batch_job("../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-10-all-attributes/20250510_113030", TEST_SET_PATH, TEST_PROMPTS)

Creating batch job for fine-tune-wdc-simple-swapping-10-all-attributes. The batch will contain 4 prompts. With 4500 pairs each.
Run configuration and input files saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-10-all-attributes/20250510_113030
Batch ID: batch_681f324e2be481909b1d998dba7ca203


In [11]:
get_batch_results_and_calculate_metrics("../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-10-all-attributes/20250510_113030")

Batch status: completed
Created at: 2025-05-10 11:02:38
Batch results saved to: ../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-10-all-attributes/20250510_113030/batch_output_batch_681f324e2be481909b1d998dba7ca203.jsonl
Metrics saved to ../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-10-all-attributes/20250510_113030/testing_stats.csv
Best performing task: 0.8140350877192982


'../../results/gpt-4.1-mini-2025-04-14/all-attributes/fine-tune-wdc-simple-swapping-10-all-attributes/20250510_113030/testing_stats.csv'